# 02 - Estadistica de negocio

Este notebook conecta el EDA con decisiones de negocio.

La idea no es solo describir datos, sino comprobar si las diferencias de margen entre canales, categorias y politicas comerciales parecen relevantes.

Preguntas que responde:

- Los marketplaces venden mas, pero dejan menos margen?
- El descuento esta reduciendo margen de forma clara?
- Hay categorias con comportamiento muy distinto?
- Existen outliers de margen que deban revisarse?
- Hay patrones temporales que puedan alimentar prediccion?

## 0. Preparacion

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from conexion_sql import read_sql
from estadistica_utils import detectar_outliers_iqr, resumen_por_grupo, matriz_correlacion, interpretar_pvalor
from visualizacion_utils import guardar_figura
from eda_utils import asegurar_directorio

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

OUT_DATOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "datos")
OUT_GRAFICOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "graficos")
OUT_CALIDAD = asegurar_directorio(PROJECT_ROOT / "outputs" / "informes_calidad")

## 1. Carga de datos

Usamos la vista enriquecida porque ya contiene las dimensiones de negocio necesarias: producto, canal, cliente, promocion, pago, envio y margen.

In [ ]:
ventas = read_sql("SELECT * FROM dbo.vw_ventas_enriquecidas")
ventas["fecha"] = pd.to_datetime(ventas["fecha"])

ventas["descuento_pct_real"] = ventas["importe_descuento"] / (ventas["unidades_vendidas"] * ventas["precio_unitario_bruto"])
ventas["peso_comision_sobre_venta"] = ventas["comision"] / ventas["ventas_netas"].replace(0, np.nan)
ventas["peso_envio_sobre_venta"] = ventas["coste_envio_asignado"] / ventas["ventas_netas"].replace(0, np.nan)
ventas["peso_coste_producto_sobre_venta"] = ventas["coste_producto"] / ventas["ventas_netas"].replace(0, np.nan)
ventas = ventas.replace([np.inf, -np.inf], np.nan)

print(ventas.shape)
display(ventas.head())

## 2. Estadistica descriptiva por canal

Logica de negocio: si dos canales tienen ventas parecidas pero margen distinto, el dashboard debe ayudar a decidir donde potenciar catalogo, negociar comisiones o ajustar precios.

In [ ]:
metricas_clave = ["ventas_netas", "margen", "margen_pct", "descuento_pct_real", "peso_comision_sobre_venta", "peso_envio_sobre_venta"]
resumen_canal = resumen_por_grupo(ventas, ["canal", "tipo_canal"], metricas_clave)
display(resumen_canal)
resumen_canal.to_csv(OUT_DATOS / "estadistica_resumen_canal.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=ventas, x="canal", y="margen_pct", ax=axes[0])
axes[0].set_title("Distribucion del margen % por canal")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylim(-0.5, 1.0)

sns.boxplot(data=ventas, x="tipo_canal", y="margen_pct", ax=axes[1])
axes[1].set_title("Margen %: venta directa vs marketplace")
axes[1].set_ylim(-0.5, 1.0)

guardar_figura("estadistica_boxplot_margen_canal.png", OUT_GRAFICOS)
plt.show()

## 3. Contraste: venta directa vs marketplace

Hipotesis de negocio:

- H0: el margen porcentual de venta directa y marketplaces no presenta diferencias relevantes.
- H1: el margen porcentual cambia segun el tipo de canal.

Usamos Mann-Whitney porque el margen por linea suele tener distribuciones asimetricas y outliers.

In [ ]:
directo = ventas.loc[ventas["tipo_canal"].str.lower().eq("direct"), "margen_pct"].dropna()
marketplace = ventas.loc[ventas["tipo_canal"].str.lower().eq("marketplace"), "margen_pct"].dropna()

resultado_mw = stats.mannwhitneyu(directo, marketplace, alternative="two-sided")
contraste_directo_marketplace = pd.DataFrame(
    {
        "comparacion": ["Direct vs Marketplace"],
        "n_direct": [len(directo)],
        "n_marketplace": [len(marketplace)],
        "mediana_direct": [directo.median()],
        "mediana_marketplace": [marketplace.median()],
        "estadistico": [resultado_mw.statistic],
        "pvalor": [resultado_mw.pvalue],
        "interpretacion": [interpretar_pvalor(resultado_mw.pvalue)],
    }
)

display(contraste_directo_marketplace)
contraste_directo_marketplace.to_csv(OUT_DATOS / "estadistica_contraste_direct_marketplace.csv", index=False)

## 4. Contraste entre canales

Ahora comprobamos si todos los canales se comportan igual o si hay diferencias globales en margen.

In [ ]:
grupos_canal = [grupo["margen_pct"].dropna() for _, grupo in ventas.groupby("canal")]
resultado_kruskal = stats.kruskal(*grupos_canal)

contraste_canales = pd.DataFrame(
    {
        "test": ["Kruskal-Wallis margen_pct por canal"],
        "estadistico": [resultado_kruskal.statistic],
        "pvalor": [resultado_kruskal.pvalue],
        "interpretacion": [interpretar_pvalor(resultado_kruskal.pvalue)],
    }
)

display(contraste_canales)
contraste_canales.to_csv(OUT_DATOS / "estadistica_contraste_canales.csv", index=False)

## 5. Correlaciones economicas

Logica de negocio: queremos detectar que variables estan mas asociadas al margen para orientar acciones: descuentos, comisiones, costes logisticos o mix de producto.

In [ ]:
columnas_corr = [
    "unidades_vendidas",
    "precio_unitario_bruto",
    "importe_descuento",
    "descuento_pct_real",
    "ventas_netas",
    "coste_producto",
    "comision",
    "coste_pago",
    "coste_envio_asignado",
    "costes_totales",
    "margen",
    "margen_pct",
]

corr = matriz_correlacion(ventas, columnas_corr)
display(corr)
corr.to_csv(OUT_DATOS / "estadistica_correlaciones.csv")

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlacion economica")
guardar_figura("estadistica_correlaciones_heatmap.png", OUT_GRAFICOS)
plt.show()

## 6. Outliers de margen

Los outliers no se eliminan automaticamente: en negocio pueden representar errores, promociones agresivas o productos que requieren una decision.

In [ ]:
ventas_outliers = detectar_outliers_iqr(ventas, "margen_pct")
resumen_outliers = (
    ventas_outliers.groupby("canal", as_index=False)
    .agg(
        lineas=("id_linea_venta", "count"),
        outliers_margen=("margen_pct_outlier_iqr", "sum"),
        margen_medio=("margen_pct", "mean"),
    )
)
resumen_outliers["pct_outliers"] = resumen_outliers["outliers_margen"] / resumen_outliers["lineas"]

display(resumen_outliers.sort_values("pct_outliers", ascending=False))
resumen_outliers.to_csv(OUT_DATOS / "estadistica_outliers_margen_por_canal.csv", index=False)

## 7. Modelo estadistico explicativo

Este no es el modelo predictivo final. Es un modelo interpretable para explicar que factores se asocian al margen %.

Sirve para la memoria porque traduce la base de datos en una explicacion de negocio.

In [ ]:
muestra_modelo = ventas[[
    "margen_pct",
    "descuento_pct_real",
    "peso_comision_sobre_venta",
    "peso_envio_sobre_venta",
    "peso_coste_producto_sobre_venta",
    "unidades_vendidas",
    "canal",
    "categoria",
]].dropna().sample(n=min(20000, len(ventas)), random_state=42)

formula = (
    "margen_pct ~ descuento_pct_real + peso_comision_sobre_venta + "
    "peso_envio_sobre_venta + peso_coste_producto_sobre_venta + "
    "unidades_vendidas + C(canal) + C(categoria)"
)
modelo_ols = smf.ols(formula=formula, data=muestra_modelo).fit()

print(modelo_ols.summary())

coeficientes = modelo_ols.params.reset_index()
coeficientes.columns = ["variable", "coeficiente"]
coeficientes.to_csv(OUT_DATOS / "estadistica_modelo_explicativo_coeficientes.csv", index=False)

## 8. Evolucion temporal

Esta parte prepara el terreno para prediccion: si hay tendencia o estacionalidad, tiene sentido usar variables temporales en ML.

In [ ]:
serie_mensual = (
    ventas.assign(periodo=ventas["fecha"].dt.to_period("M").dt.to_timestamp())
    .groupby("periodo", as_index=False)
    .agg(
        ventas_netas=("ventas_netas", "sum"),
        margen=("margen", "sum"),
        unidades=("unidades_vendidas", "sum"),
    )
)
serie_mensual["margen_pct"] = serie_mensual["margen"] / serie_mensual["ventas_netas"]
serie_mensual["ventas_media_3m"] = serie_mensual["ventas_netas"].rolling(3, min_periods=1).mean()

display(serie_mensual)
serie_mensual.to_csv(OUT_DATOS / "estadistica_serie_mensual.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=serie_mensual, x="periodo", y="ventas_netas", marker="o", label="Ventas netas", ax=ax)
sns.lineplot(data=serie_mensual, x="periodo", y="ventas_media_3m", marker="o", label="Media movil 3m", ax=ax)
ax.set_title("Evolucion mensual de ventas netas")
ax.tick_params(axis="x", rotation=45)
guardar_figura("estadistica_evolucion_mensual.png", OUT_GRAFICOS)
plt.show()

## 9. Conclusiones para la memoria

Despues de ejecutar, completar con resultados concretos:

- Que canal deja mayor margen relativo y cual genera mas volumen.
- Si marketplace tiene una diferencia estadistica frente a venta directa.
- Que costes pesan mas sobre la rentabilidad.
- Que outliers merecen revision de precio, descuento o coste.
- Que patrones temporales justifican pasar a machine learning.